# AI-assisted schema markup creation

#### The plan:

- Crawl a website
- Select certain pages where we know their content and structure
- Determine the schema type and properties that can be found on those pages
- Send bulk requests to an LLM and get the structured data

## Import packages

In [ ]:
import os

import advertools as adv
import pandas as pd
from openai import OpenAI

pd.options.display.max_columns = None

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

## Crawl a website

In [ ]:
adv.crawl(
    url_list="https://searchnstuff.co.uk/",
    output_file="searchnstuff.jsonl",
    follow_links=True,
)

## Get the event URLs from the `searchnstuff.co.uk` website:

- Read the crawl file
- Split the URLs using `adv.url_to_df`

In [4]:
crawldf = pd.read_json("searchnstuff.jsonl", lines=True)
urldf = adv.url_to_df(crawldf["url"].str.replace(r"./", ""))
urldf.sample(10)

,url,scheme,netloc,path,query,fragment,dir_1,dir_2,last_dir
17,https://searchnstuff.co.uk/workshops/,https,searchnstuff.co.uk,/workshops/,None,None,workshops,None,workshops
21,https://searchnstuff.co.uk/networking-dinners/...,https,searchnstuff.co.uk,/networking-dinners/come-together-search-n-stu...,None,None,networking-dinners,come-together-search-n-stuff-community-christm...,come-together-search-n-stuff-community-christm...
22,https://searchnstuff.co.uk/meetups/search-n-st...,https,searchnstuff.co.uk,/meetups/search-n-stuff-london-meetup-event-hi...,None,None,meetups,search-n-stuff-london-meetup-event-highlights,search-n-stuff-london-meetup-event-highlights
10,https://searchnstuff.co.uk/join-us/,https,searchnstuff.co.uk,/join-us/,None,None,join-us,None,join-us
18,https://searchnstuff.co.uk/meetups/search-n-st...,https,searchnstuff.co.uk,/meetups/search-n-stuff-antalya-meetup-digital...,None,None,meetups,search-n-stuff-antalya-meetup-digital-marketin...,search-n-stuff-antalya-meetup-digital-marketin...
12,https://searchnstuff.co.uk/conferences/search-...,https,searchnstuff.co.uk,/conferences/search-n-stuff-antalya-conference...,None,None,conferences,search-n-stuff-antalya-conference-2024,search-n-stuff-antalya-conference-2024
9,https://searchnstuff.co.uk/about-us/,https,searchnstuff.co.uk,/about-us/,None,None,about-us,None,about-us
11,https://searchnstuff.co.uk/speakers-hub/,https,searchnstuff.co.uk,/speakers-hub/,None,None,speakers-hub,None,speakers-hub
4,https://searchnstuff.co.uk/privacy-policy/,https,searchnstuff.co.uk,/privacy-policy/,None,None,privacy-policy,None,privacy-policy
6,https://searchnstuff.co.uk/meetups/,https,searchnstuff.co.uk,/meetups/,None,None,meetups,None,meetups


## Find event pages
- Create a filter getting URLs where `dir_1` is in `["networking-dinners", "meetups", "conferences", "workshops"]`
- Save to a variable called `event_urls`

In [5]:
event_urls = urldf[
    urldf["dir_1"].isin(["networking-dinners", "meetups", "conferences", "workshops"])
    & urldf["dir_2"].notna()
]["url"].tolist()


In [6]:
event_urls

['https://searchnstuff.co.uk/conferences/search-n-stuff-antalya-conference-2025/',
 'https://searchnstuff.co.uk/conferences/search-n-stuff-antalya-conference-2024/',
 'https://searchnstuff.co.uk/meetups/search-n-stuff-london-meetup-in-partnership-with-semrush/',
 'https://searchnstuff.co.uk/meetups/search-n-stuff-istanbul-edition/',
 'https://searchnstuff.co.uk/meetups/search-n-stuff-london-meetup-summer-party-networking/',
 'https://searchnstuff.co.uk/meetups/search-n-stuff-antalya-meetup-digital-marketing-tips-startup-growth/',
 'https://searchnstuff.co.uk/meetups/search-n-stuff-london-meetup-in-partnership-with-studiohawk/',
 'https://searchnstuff.co.uk/meetups/search-n-stuff-x-podcast-growth-insights-strategies-and-real-stories/',
 'https://searchnstuff.co.uk/networking-dinners/come-together-search-n-stuff-community-christmas-dinner/',
 'https://searchnstuff.co.uk/meetups/search-n-stuff-london-meetup-event-highlights/']

## Create a function to generate JSON-LD

In [7]:
def generate_schema(crawl_df, schema_type, *properties):
    """Request the schema for a given crawl Dataframe, a schema type and any properties

    Parameters
    ----------
    crawl_df : pandas.DataFrame
        A crawl DataFrame generated from the advertools crawl function
    schema_type : str
        The schema.org type to use (Person, Organization, Event, etc.)
    properties : str
        One or more properties to extract from the given pages (price, currency, etc.)
    """
    responses = []
    for _, row in crawl_df.iterrows():
        completion = client.chat.completions.create(
            model="gpt-4o",
            temperature=0,
            seed=123,
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": """
                You are an expert in structured data, especially JSON-LD, you respond in
                JSON only.
                You provide the details of the requested properties only.
                In case you cannot extract the requied data, you respond with `null`""",
                },
                {
                    "role": "user",
                    "content": f"""
        Please create the JSON-LD schema using the following details from the landing page:

        "@type": {schema_type}

        - required schema properties: {", ".join(properties)}

        - Details from landing page:

        HTML title: {row.get("title", "NOT AVAILABLE - PLEASE IGNORE")}
        HTML meta description: {row.get("meta_desc", "NOT AVAILABLE - PLEASE IGNORE")}
        HTML h1 tags: {row.get("h1", "NOT AVAILABLE - PLEASE IGNORE")}
        HTML h2 tags: {row.get("h2", "NOT AVAILABLE - PLEASE IGNORE")}
        HTML body text: {row.get("body_text", "NOT AVAILABLE - PLEASE IGNORE")}
        """,
                },
            ],
        )
        responses.append((row["url"], completion))
    return responses


In [8]:
event_df = crawldf[crawldf["url"].isin(event_urls)]

In [ ]:
responses = generate_schema(
    event_df,
    "BusinessEvent",
    "name",
    "description",
    "sartDate",
    "endDate",
    "organizer",
    "isAccessibleForFree",
    "location",
)

In [ ]:
import json

schema_df = pd.concat(
    [pd.json_normalize(json.loads(r[1].choices[0].message.content)) for r in responses]
)
schema_df.insert(0, "url", [r[0] for r in responses])
schema_df.to_csv("schema_df.csv", index=False)

In [11]:
schema_df = pd.read_csv("schema_df.csv")
schema_df

,url,@context,@type,name,description,startDate,endDate,isAccessibleForFree,organizer.@type,organizer.name,organizer.url,location.@type,location.name,location.address.@type,location.address.addressLocality,location.address.addressCountry,location.address.streetAddress,location.address.postalCode,location,location.address.addressRegion,location.address
0,https://searchnstuff.co.uk/conferences/search-...,https://schema.org,BusinessEvent,Search ‘n Stuff Antalya Global Digital Marketi...,Join us for a 5-Star Luxurious Learning & Netw...,2025-10-09,2025-10-12,False,Organization,Search 'n Stuff,https://searchnstuff.co.uk,Place,Baia Lara Hotel,PostalAddress,Antalya,Turkey,NaN,NaN,NaN,NaN,NaN
1,https://searchnstuff.co.uk/conferences/search-...,https://schema.org,BusinessEvent,Search ‘n Stuff Antalya Conference 2024,Welcome to the inaugural Search ‘n Stuff Confe...,NaN,NaN,NaN,Organization,Yagmur Simsek,NaN,Place,"Antalya, Turkey",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://searchnstuff.co.uk/meetups/search-n-st...,https://schema.org,BusinessEvent,Search ‘n Stuff London Meetup – in partnership...,Ready for something epic? We’re teaming up wit...,2024-03-18T19:00:00+00:00,2024-03-18T21:30:00+00:00,False,Organization,Search ‘n Stuff,https://www.linkedin.com/company/search-n-stuff/,Place,Paddington Works,PostalAddress,London,UK,8 Hermitage St,W2 1BE,NaN,NaN,NaN
3,https://searchnstuff.co.uk/meetups/search-n-st...,https://schema.org,BusinessEvent,Search ‘n Stuff Istanbul Edition,Search ‘n Stuff Istanbul Edition on October 15...,2023-10-15,2023-10-15,False,Organization,searchnstuff.co.uk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://searchnstuff.co.uk/meetups/search-n-st...,https://schema.org,BusinessEvent,Search ‘n Stuff London Meetup – Summer Party &...,"Get ready, London! The biggest Search ‘n Stuff...",2025-06-11T18:30:00+01:00,2025-06-11T22:00:00+01:00,False,Organization,Search ‘n Stuff,NaN,Place,Doggett’s Coat and Badge,PostalAddress,London,UK,NaN,NaN,NaN,NaN,NaN
5,https://searchnstuff.co.uk/meetups/search-n-st...,https://schema.org,BusinessEvent,Search ‘n Stuff Antalya Meetup – Digital Marke...,Antalya’s digital marketing and startup scene ...,2025-03-21T18:00:00+03:00,2025-03-21T22:00:00+03:00,False,Organization,Search 'n Stuff Community,NaN,Place,The Soul Antalya,PostalAddress,Muratpaşa,Turkey,Kılınçarslan Mh. Park Sk. No:36 Thesoul,07010,NaN,Antalya,NaN
6,https://searchnstuff.co.uk/meetups/search-n-st...,https://schema.org,BusinessEvent,Search ‘n Stuff London Meetup – in partnership...,Join us for our the first Search ‘n Stuff even...,2024-01-25T18:00:00+00:00,2024-01-25T21:30:00+00:00,False,Organization,Search ‘n Stuff,https://www.linkedin.com/company/search-n-stuff/,Place,Looking Glass Cocktail Club,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"49 Hackney Rd, Shoreditch, London"
7,https://searchnstuff.co.uk/meetups/search-n-st...,https://schema.org,BusinessEvent,"Search ‘n Stuff x Podcast Growth: Insights, St...",Join us for Search 'n Stuff x Podcast Growth E...,2025-01-15T18:30,2025-01-15T22:00,False,Organization,Search 'n Stuff,NaN,Place,Linden House,PostalAddress,London,UK,Hammersmith,W6 9TA,NaN,NaN,NaN
8,https://searchnstuff.co.uk/networking-dinners/...,https://schema.org,BusinessEvent,Come Together: Search ‘n Stuff Community Chris...,"As the year draws to a close, we're creating a...",2024-12-12T20:00:00,NaN,False,Organization,Search 'n Stuff,NaN,Place,The Counter Soho,PostalAddress,London,NaN,NaN,NaN,NaN,NaN,NaN
9,https://searchnstuff.co.uk/meetups/search-n-st...,https://schema.org,BusinessEvent,Search ‘n Stuff London Meetup – Event Highlights,The first Search ‘n Stuff meetup of 2024 on Ja...,2024-01-25,2024-01-25,False,Organization,Search ‘n Stuff,https://searchnstuff.co.uk,Place,London,PostalAddress,London,UK,NaN,NaN,NaN,NaN,NaN


In [ ]:
import plotly.express as px

fig = px.scatter(
    schema_df,
    x="startDate",
    y=[1 for i in range(len(schema_df))],
    hover_name="name",
    opacity=0.6,
    title="Search 'n Stuff Event Timeline",
    template="cosmo",
    hover_data=["name", "startDate"],
    height=350,
    custom_data=["name", "startDate"],
    size=[14 for i in range(len(schema_df))],
    labels={"startDate": "Event date"},
)
fig.layout.yaxis.visible = False
fig.data[0].hovertemplate = "<b>%{customdata[0]}</b><br><br>Date: %{customdata[1]}"

fig

## Formatting and structuring the inputs for the LLM

In [90]:
from IPython.display import display_markdown


def dm(text):
    return display_markdown(text, raw=True)


for _, row in crawldf[crawldf["url"].str.contains("2025")].iterrows():
    dm(f"#### **Title:** {row.title}")
    dm(f"#### **Meta description:** {row.meta_desc}")
    dm(f"#### **h1:** {row.h1}")
    dm(f"#### **h2:** {row.h2.split('@@')}")
    dm(f"#### **h3:** {row.h3.split('@@')}")
    dm(f"#### **h4:** {row.h4.split('@@')}")
    dm(f"#### **Body text:** {row.body_text[109:]}")

#### **Title:** Search 'n Stuff Antalya Global Digital Marketing Conference 2025 - searchnstuff.co.uk

#### **Meta description:** Search 'n Stuff Antalya Global Digital Marketing Conference 2025: Perfect for marketing enthusiasts, experts, innovators. Join us!

#### **h1:** Search ‘n Stuff Antalya Global Digital Marketing Conference 2025

#### **h2:** ['Your Digital Journey in Progress – Connect, Learn & Grow in a 5-Star Setting', 'Why Should You Be There?', 'What’s Included?', 'Secure Your Spot – Limited Tickets Available!', 'Conference Overview', 'Our Speakers (A-Z)', 'Our Sponsors', 'Calling All Sponsors – Amplify Your Brand with Search ‘n Stuff!', 'Why Should You Attend? Convince Your Manager or Colleagues!', 'Frequently Asked Questions', 'About the Venue', 'Subscribe to the newsletter to be the first to be informed!', 'About Us', 'Write An E-mail', 'Address']

#### **h3:** ['All Tickets Include', 'Early Bird Tickets – June 1, 2025 – July 31, 2025', 'Platinum Sponsors', 'Community Supporters', 'Media Partners', 'Workshop Sponsors', '\nRelated Posts']

#### **h4:** ['Conference Details', 'Simultaneous Translation', 'Transformative Knowledge', 'Unparalleled Networking', 'ExclusiveExperience', 'UnbeatableValue', 'Luxurious Accommodation', 'All-Inclusive Dining and Drinks', 'Conference and Workshops', 'Stunning Beachside Setting', 'Baia Lara Hotel', 'Search ‘n Stuff Antalya Conference 2024']

#### **Body text:** 
 Search ‘n Stuff Antalya Global Digital Marketing Conference 2025 Join Us for a 5-Star Luxurious Learning & Networking Getaway! Your Digital Journey in Progress – Connect, Learn & Grow in a 5-Star Setting After an incredible first edition, Search ‘n Stuff Antalya is back—bigger, bolder, and packed with even more insights, connections, and experiences! This is not just another conference; it’s a movement uniting the global digital marketing and search community in a stunning 5-star, all-inclusive setting. From SEO, content, CRO, and analytics to AI-driven marketing, eCommerce growth, and paid search, we cover it all with world-class speakers, actionable workshops, and unmissable networking opportunities. Conference Details 🗓️  Date :  October 9-12, 2025 📍  Venue :   Baia Lara Hotel, Antalya, Türkiye (Turkey) BUY EARLY BIRD TICKETS DON’T MISS THE OPPORTUNITY VIEW SPEAKERS VIEW SPONSORS Why Should You Be There? Gain insights from  25+ international speakers , and  15+ workshops  tailored for professionals at every level. Engage in dynamic networking events like speed networking, PowerPoint karaoke, and live podcast sessions. Connect with 250+ industry peers from across the globe in an environment designed for both learning and relaxation. (SEO, Content, CRO & UX) 250+ Attendees from all over the world. Simultaneous Translation All speeches to be given on the main stage will be translated with simultaneous translation (Turkish < > English). Transformative Knowledge Learn from global experts in digital marketing and business growth. Gain actionable insights from industry-leading speakers. Unparalleled Networking Connect with global industry leaders and peers in an inspirational, professional setting. Exclusive Experience Limited availability ensures a personalized and impactful event. Unbeatable Value Enjoy luxury, knowledge, and connection for a fraction of the price. Luxuriate in the comfort of the Baia Lara Hotel, a beachfront haven. What’s Included? By purchasing a  Early Bird Ticket , you can benefit from all the opportunities below.  Don’t miss the early price advantage! Luxurious Accommodation 3 nights at Baia Lara Hotel (5-star luxury) Check-in & out: October 9-12 All-Inclusive Dining and Drinks Daily breakfast, lunch, and dinner Afternoon snacks by the pool All-day open bar (alcoholic & nonalcoholic drinks) Conference and Workshops Access to conference talks and workshops on: Organic Growth (SEO, Content, CRO & UX) Paid Growth (PPC, Social Ads, Measurement) Data & Analytics Networking events and activities Stunning Beachside Setting The pristine Mediterranean coast of Antalya, offering relaxation and inspiration in equal measure. Secure Your Spot – Limited Tickets Available! Don’t miss out—grab your ticket now before prices rise! Details: Before your make your decision, there are two types of conference tickets: “tickets with accommodation” and “tickets with no accommodation/only access to conference days” All Tickets Include Access to all conference talks & workshops (Oct 10-11) Topics:  SEO, Content, PPC, Social Ads, CRO, UX, Data, and Analytics Lunch & coffee breaks on conference days Networking events & activities Search ‘n Stuff Goodie Bag Tickets with Accommodation (single room, double room, group tickets)  also include: 3 Nights at 5 Baia Lara Hotel (Oct 9-12) with Breakfast, Lunch, Dinner & Open Bar Early Bird Tickets –  June 1, 2025 – July 31, 2025 Conference Only No Accommodation available €192 €208 Single Room Ticket 1 Attendee – Private Room €774 €968 Double Room  Ticket 2 Attendees – Shared Room €1,044 €1,305 Group Ticket (Double Rooms) 6 Attendees – Shared Rooms €2,819 €3,524 Group Ticket (Single Rooms) 6 Attendees – Private Rooms €4,180 €5,225 Conference Overview Day 1: October 9, 2025 Arrival & Check-in Evening Networking Event Day 2: October 10, 2025 Main Conference Talks Evening Socials Day 3: October 11, 2025 Workshops in Various Verticals Networking & After-Party Day 4: October 12, 2025 Check-out & Farewell Our Speakers (A-Z) Aleyda Solis SEO Consultant and Founder @ Orainti & SEOFOMO Aleyda Solis is an SEO consultant and founder of Orainti -a boutique SEO consultancy advising top brands worldwide-, speaker and author. She shares the latest news and resources in SEO in the SEOFOMO News, the SEOFOMO Newsletter and Digital Marketing in MarketingFOMO, organizes the SEOFOMO Meetup series, provides SEO tips in the Crawling Mondays video series, and a free SEO Learning Roadmap called LearningSEO.io. Visit LinkedIn Profile Andy Chadwick Co-founder at Snippet Digital & Keyword Insights Andy Chadwick has spent the best part of a decade in SEO, working both in-house and agency-side before co-founding Snippet Digital and Keyword Insights—a SaaS tool designed to help businesses become topical authorities. More than just an SEO, Andy is an entrepreneur who has built multiple businesses to seven figures, blending data-driven strategy with innovation to drive growth. Visit LinkedIn Profile Arnout Hellemans Freelance Tech SEO and Analytics Consultant at Onlinemarkethink.com Arnout Hellemans is a Dutch tech SEO and online strategy consultant who has been freelancing since 2009. And has been working in e-commerce, publishing, B2B, and lead gen companies on their SEO / CRO and Strategy. He mostly works in the UK and US market at this moment even though he is based in Amsterdam. He has shared his insights and findings at loads of conferences and has appeared on quite a few podcasts. He is driven by making the web a less frustrating place. Visit LinkedIn Profile BAKI ERFIDAN Regional Manager MENA & Turkey AT WHITEPRESS Arnout Hellemans is a Dutch tech SEO and online strategy consultant who has been freelancing since 2009. And has been working in e-commerce, publishing, B2B, and lead gen companies on their SEO / CRO and Strategy. He mostly works in the UK and US market at this moment even though he is based in Amsterdam. He has shared his insights and findings at loads of conferences and has appeared on quite a few podcasts. He is driven by making the web a less frustrating place. Visit LinkedIn Profile BASAK ODEMIS Founder & Growth Consultant @ Make Digital Agency Basak is a Digital Marketing Consultant and Founder of Make Digital Agency, with 7 years of experience in PPC and paid social. She has worked with renowned brands like Saatchi Gallery, Deliveroo, Getir, Philips and Dyson, driving growth through data-driven strategies. Basak specializes in setting digital marketing direction, managing high-performing ad campaigns, and leveraging advanced analytics to optimize performance. Visit LinkedIn Profile BEGUM KAYA SEO Consultant Once an international SEO nomad, Begum made the leap to being an SEO Manager at Uprise Up. Now, happily ‘converted’ back to the agency realm, she is helping charities develop genuinely effective strategies with minimal fuss. Begüm also moderates the Opinionated SEO Opinions web series, a collaborative venture with The Gray Dot Company, where she shares insights and experiences and gets to talk to brilliant hosts and guests who are enriching the industry. It was the active and welcoming sense of community that first attracted her to SEO, so it’s her personal mission to both learn from and contribute to that community – she has spoken at a number of industry conferences, including Brighton SEO and SERPConf. Beyond her professional life, she’s a passionate yogini, which means she’s flexible even when her calendar isn’t. Visit LinkedIn Profile Christoph Kottmann Search & Analytics Consultant / Freelancer & Moccu Agency Christoph Kottmann is a freelance SEO and web analytics consultant specializing in e-commerce and growth businesses. His career path took him from journalism to content creation and finally to analytical search engine optimization. He helps B2B and B2C companies with website relaunches, content strategies, and data-driven optimization. Visit LinkedIn Profile Clara Soteras SEO for News Publishers Consultant and Head of Innovation & Digital Strategy @ AMIC Clara Soteras is an SEO for News Publishers and Digital Strategy Consultant. She is currently Head of Innovation and Digital Strategy at AMIC, a local media association with more than 600 members, focused on new projects that incorporate artificial intelligence. Her areas of expertise include attracting new audiences, Google Discover, content planning, SEO functions in a newsroom and business-focused editorial product. She is also an associate professor at the Autonomous University of Barcelona and teaches SEO for News at other business schools. She is part of the judging panel for different search awards such as the Global, European or UK Search Awards and has been a member and judge for the Online News Association, WAN-IFRA and LION Publishers. She is also one of the expert contributors to the evaluation of the IA Act code of practice of the European Commission’s IA Office. She has been the Director of SEO and Product at El Nacional, a Spanish media company, and SEO Manager at betevé, a local TV channel in Barcelona. She actively participates in SEO, AI, journalism and media conferences, giving presentations, round tables and debates with other experts in the sector. She is part of the #MujeresEnSEO community and the SEO association in Catalan, as well as a member of Google Product Experts, where she shares knowledge with other professionals. Visit LinkedIn Profile DAWN ANDERSON Founder & SEO Consultant at Bertey Dawn Anderson is the founder of SEO consultancy and digital marketing agency, Bertey, based in Manchester, UK. Dawn is an international conference speaker (UK, US, AUS, IT, FR, ES), covering advanced technical SEO and digital marketing topics. Dawn has spoken at some of the leading global search marketing conferences, including Pubcon (Las Vegas and Florida), SASCON, Brighton SEO, SMX (London (UK), San Jose (US), Milan (Italy), MozCon (US) and State of Search (US). Dawn is qualified in digital marketing strategy with both a Pg Dip DigM (Post Graduate Diploma) (IDM) and a Master of Science (MSc) in (Digital Marketing Strategy). Dawn is a Fellow of The Institute of Digital Marketing (FIDM) and a Member of The British Computing Society (MBCS). She also contributes to leading search marketing industry publications such as Search Engine Land, Search Engine Journal, The SEMPost, State of Digital, and Smart Insights. Dawn is currently studying for her second MSc in Computer Science and Data Science. Visit LinkedIn Profile Defne Babat Strategy & Growth Manager @ Karaca Defne Babat is a digital marketing and growth strategist who has worked with some of Turkey’s leading e-commerce companies, focusing on performance marketing, SEO, app growth, and full-funnel optimization. She currently leads strategic marketing initiatives as part of the CEO Office at Karaca Group, overseeing multi-brand and multi-market digital operations. Her approach centers on data-driven decision-making, marketing efficiency, and delivering measurable business outcomes. Visit LinkedIn Profile Dixon Jones CEO @ Waikay & InLinks Dixon Jones, a respected veteran in SEO since 1999, is the CEO of InLinks.net, focusing on entity-based optimization. He was instrumental in his former role as Marketing Director at Majestic. A renowned international speaker and thought leader, Dixon brings deep expertise in search marketing and data analysis. Visit LinkedIn Profile Elias Dabbas Creator at advertools Elias works at the intersection of SEO/SEM, data science and software development. He is the creator and maintainer of advertools, a Python package and command-line tool for digital marketing people, which has been downloaded more than 3.5 million times. Among other things, advertools provides an SEO crawler, tools for creating search ads, log file analysis, XML sitemaps, robots tools, and more. He is also the author of the book Interactive Dashboards and Data Apps with Plotly and Dash. Visit LinkedIn Profile Ema Fulga Founder. Copywriter. AI Content Strategist @ decipher Ema Fulga is a freelance copywriter and the founder of decipher., a UK-based creative agency dedicated to brands with strong personalities. Over the past seven years, she has collaborated with companies such as Publicis, the World Intellectual Property Organization, NTT DATA, the Covenant House New Jersey, and Grupo Planeta, among others. Beyond decipher., Ema is publishing her first children’s book, completing her debut historical novel, and co-owning a beauty import and distribution business. Visit LinkedIn Profile  Emma-Jane Stogdon Senior SEO Content Specialist @ Wise Emma-Jane Stogdon is Senior SEO Content Specialist at Wise, a global technology company, building the best way to move and manage the world’s money. As part of the Wise SEO team, she produces strategic content for the UK blog, on topics such as personal finance, global travel and international living. Drawing on 5+ years of SEO experience and over 15 years as a copywriter and filmmaker, Emma-Jane excels at crafting creative content ideas that achieve significant search visibility. Visit LinkedIn Profile Erkut Kose SEO Manager @ 3GEN Digital Growth Agency Erkut Köse is a results-driven SEO professional with over 7 years of expertise in the organic growth arena. He currently serves as SEO Manager at 3GEN Digital Growth Agency, where he orchestrates organic search strategies, technical optimizations, and scalable solutions to elevate brand visibility. Throughout his career, he has collaborated with industry leaders such as BtcTurk, Koçtaş, Porland, Otelz and Sportive, driving transformative SEO campaigns that deliver measurable business impact.  Visit LinkedIn Profile Eyüp Alikilic Team Lead SEO @ ATP Autoteile With nearly 10 years of specialized experience in technical SEO, he has focused exclusively (and fortunately) on eCommerce business models, gaining a deep expertise in the technical foundations required for a successful SEO strategy. His experience spans from optimizing international eCommerce platforms struggling with complex technical challenges to scaling up existing online shops across multiple markets. Visit LinkedIn Profile Gabi Toxler SEO Consultant @ Sirius Works Gabi Troxler is the co-founder and former CCO & SEO lead of feey.ch, an award-winning Swiss ecommerce startup. Prior to launching her own venture, she drove organic growth for large enterprises in healthcare, finance, and HR. Now an indepdendent SEO consultant, trainer, and speaker, Gabi helps businesses leverage content and SEO to build thriving customer relationships. Outside work, she enjoys reading fantasy and sci-fi, sports, and learning about birds and astrophysics. Visit LinkedIn Profile Gerry White SEO Consultant @ Dergal.co.uk  With over two decades in the industry, Gerry White is a seasoned SEO & Growth Consultant with a background in development and technical marketing. His expertise spans SEO, analytics, and digital growth, benefiting companies, agencies, government entities, and major corporations like the BBC.  A recognized thought leader, Gerry has spoken at top conferences worldwide, including in Paris, Denmark, the USA, and the UK. He has also served as a judge for prestigious awards like the UK Search Awards, Growth Awards, and E-Commerce Awards. His career includes leading technical SEO at Just Eat (FTSE 100) across 12 markets, directing SEO at Rise at Seven during its rapid expansion, and heading SEO at Oda, an international supermarket. In 2023–2024, he became VP of Growth at Mirador Local, an SEO tool helping businesses manage Google Business Profiles at scale. In 2025, Gerry returned to freelance consulting. As a co-founder of Take It Offline unconference events, he remains committed to industry collaboration. With a deep passion for digital marketing and a track record of driving growth, Gerry continues to be a leading voice in SEO and technical marketing. Visit LinkedIn Profile Giulia Panozzo Consultant @ Neuroscientive Giulia is an in-house Director of Customer Acquisition and a freelance neuromarketing consultant, with expertise in technical SEO and international strategy for e-commerce. She previously worked on global organic growth for websites of the caliber of RS Components and Expedia.Before landing on digital marketing, Giulia obtained a MSc in Cognitive Neuroscience and Clinical Neuropsychology and worked in academic research published internationally. She now leverages her background in Neuroscience research to drive customer acquisition tests informed by cognitive patterns and bias, and has recently launched her own consultancy, Neuroscientive.Giulia is a regular speaker at international industry conferences, where she talks about customer behaviour, search and data analytics. Visit LinkedIn Profile GOKCE YESILBAS Senior Paid Media Consultant @Vervaunt Gokce is a data-driven performance marketer with over 4 years of experience in increasing revenue and ROAS through cross-channel campaigns. She is currently pursuing her MSc in Business Intelligence and Digital Marketing at Brunel University London, where she holds an International Excellence Scholarship. Gokce has diverse experience in the digital marketing landscape, having worked as a Digital Marketing Specialist at Beymen Group, one of Turkey’s largest luxury e-commerce marketplaces, as well as a Senior Digital Marketing Specialist at Zeo Agency, an international digital marketing agency. In addition to her professional experience, Gokce is also the curator of a LinkedIn newsletter, “All About Digital Marketing & Data Analytics,” which serves as a go-to source for all things related to digital marketing and data analytics. Visit LinkedIn Profile Gus Pelogia Sr SEO Product Manager at INDEED Gus Pelogia is a journalist turned SEO professional, currently a Senior SEO Product Manager at Indeed, the #1 job site in the world. He spoke at events such as BrightonSEO, LondonSEOXL and Wolfgang Essentials. Gus is also a contributor to Moz, Wix and other well-known industry blogs. Working in cross-functional teams including content creators, UX designers, engineers, data scientists, product managers and other teams, he aims to make SEO accessible and easy to understand. His work is focused on impact, buy-in and processes from ideation to release and measurement, avoiding staying in the SEO bubble. He’s from Brazil but built all his SEO career abroad, working in-house and for agencies in Argentina, The Netherlands, and Ireland since 2012. His work has awarded him and his clients industry awards such as The Drum Search Awards and Irish Content Marketing Awards. A frequent guest in SEO podcasts, Gus has been invited to talk about SEO & Product in shows, such as Crawling Mondays, The SEO Sprint, Majestic’s SEO in 2022/2023/2024 series and many others. You can also find his articles on guspelogia.com. From link building to migrations, local and enterprise, Gus has done a bit of everything in SEO. Prior to his marketing career, he worked for some of the largest media outlets in Brazi. One of his most successful ventures was a blog on MTV, and his book Diário de Palco, where he profiled ten people involved in the independent rock scene in Brazil. Visit LinkedIn Profile Itamar Blauer  Senior SEO Director at StudioHawk Itamar Blauer is the senior SEO director at StudioHawk, a specialist SEO agency. He is an SEO trainer, speaker, author, and host of the “SEO Unplugged” podcast, sharing tips and case studies across various SEO topics. Catering to both SMEs and large enterprises within diverse B2B and B2C sectors, Itamar has a proven track record of increasing rankings with SEO that is UX-focused, data-backed, and creative. Visit LinkedIn Profile  James Brockbank Managing Director & Founder at Digitaloft James Brockbank is an experienced SEO professional and is the Managing Director & Founder at Digitaloft, a UK-based agency, and an experienced specialist in content-first SEO and digital PR. With more than 14 years’ experience in SEO, a career spanning both the technical and creative sides of the industry, and having spent the last nine years growing an agency at the forefront of the digital PR and content-led SEO space, James offers valuable insights into proven tactics as well as being an advocate for the importance of building a positive agency culture. James has previously spoken at or written for events and publications including SMX, BrightonSEO, Pubcon, WordCamp, State of Search, Search Engine Journal, Search Engine Land and Semrush. Visit LinkedIn Profile JUDITH LEWIS Founder at Decabbit Consultancy Judith is a renowned international MC, keynote speaker, writer, trainer, blogger and digital media consultant specialising in applying strategic understanding of digital technologies to help businesses innovate and optimise their effectiveness within the new, networked communications environment. Her honours degree in psychology, combined with her background in programming and in law has made her uniquely insightful when dealing with companies of different sizes. She is a regular speaker and trainer around the world on SEO, content strategy, link building, paid media, and digital strategy, and has been recognised by her peers as one of the most influential people in the UK digital industry. Her skills in digital marketing, as well as her business-mindedness, gives her a unique insight when consulting for businesses. Judith judges the industry leading Search Awards around the world, and has every year since their inception. She has worked with market-leading global businesses including Google, NatWest/RBS, National Gallery, Fidelity, GalaCoral, NBC Universal, Readers Digest, Bayer, Amadeus, AMD, AmEx, Virgin.com, Virgin Startup, LoveToKnow, Wicked Uncle, Zopa &amp; more. She has over 25 years of experience, predating Google in her entry into the market, and now runs her own consultancy. She has worked both in-house and agency-side. Judith also blogs about wine, tourism, and chocolate, has contributed to books on SEO, and wrote the Econsultancy Best Practice Guide to SEO. Visit LinkedIn Profile Kyle Rushton McGregor Director @ KRM Digital Marketing Ltd Kyle is an experienced GA4 and Analytics consultant with over a decade of expertise. As the founder of KRM Digital Marketing Ltd., he brings valuable insights from both agency and client-side roles. Throughout his career, Kyle has worked with a wide range of organisations, including charities, B2B, B2C, and everything in between. Visit LinkedIn Profile Marcel Schröder Search & Data Hacker, Freelancer Marcel Schröder is a search & data hacker who’s working for over 10 years as a freelancer in SEO, analytics, and performance. From startups to global players, he lives by the motto: ‘Study the data, then trust your gut’. Beyond data, he’s passionate about nature, good food and all things music & podcasts. Visit LinkedIn Profile Mark Williams-Cook Digital Marketing Director @ Candour Mark Williams-Cook has over 20 years of SEO experience and is co-owner of search agency Candour, the founder of AlsoAsked and runs multiple other websites and a UK e-commerce company. Outside of speaking at conferences, Mark has trained over 8,000 SEOs with his Udemy and in-person course, runs the Core Updates SEO newsletter, organises the annual SearchNorwichXL conference and finds security bugs for sport. Visit LinkedIn Profile  Martin Splitt Search Relations Engineer @ Google Martin helps humans make computers beep & boop the way they’re supposed to. Mostly to do with websites, really. Whenever he doesn’t do that, he’s underwater. Visit LinkedIn Profile MELISSA SAFAK Senior Marketing Executive @ BVA BDRC Berfin Melissa Şafak is a seasoned marketing professional with a specialized focus on tech and product marketing within the SaaS industry. Currently serving as a Marketing Executive at fu3e. in London, Melissa has led impactful branding projects and executed comprehensive product marketing campaigns that have significantly boosted brand awareness and lead generation. Melissa holds a Master’s degree in International Marketing from King’s College London and a Bachelor’s degree in International Relations and Development from the University of Westminster. With over two years of experience, she has successfully driven business growth through strategic marketing initiatives. Her expertise spans demand generation, product lifecycle management, and marketing automation. Known for her data-driven approach, Melissa helps businesses optimize their digital presence and implement effective product marketing strategies. Visit LinkedIn Profile MERT ERKAL Founder & Visionary at Stradiji.com He is the founder of Stradiji, which has provided Search Engine Optimization (SEO), Conversion Optimization and Digital Ads consulting services since 2010. Stradiji creates digital marketing strategies for companies of all sizes, enables them to be implemented and increases the awareness and influence of the companies they serve in the online and offline world. Visit LinkedIn Profile MURAT YATAGAN Strategic Growth Advisor A veteran of Google’s core search departments, I’ve driven explosive growth at startups like Brainly, scaling it to 400M monthly users, and Global Savings Group. Founded my SEO consultancy a decade ago, I’ve delivered organic growth advisory to 100+ companies, while also shaping industry standards as a Judge for Search Awards across Europe, the US, MENA, and the UK. As a Google startup acceleration mentor and a current Stanford leadership student, I’m committed to continuous growth and innovation in the digital sphere. Visit LinkedIn Profile OZAN KETENCI Generative AI Strategist, Zeo Agency Ozan Ketenci is a Generative AI Strategist from London, specializing in leveraging advanced technology to scale SEO and digital marketing strategies. At Zeo, where he leads as a strategist, he has pioneered the integration of generative AI technologies, enhancing business operations and elevating client engagement across diverse industries. His career in SEO spans over a decade, during which he has escalated from an SEO Analyst to leading strategy and consulting, directing a team of over 30 professionals. His work focuses on employing cutting-edge AI tools to deepen analytical capabilities, refine web strategies, and optimize search marketing, ensuring clients achieve top search engine rankings and maximized digital presence. I am deeply committed to educating and developing others, regularly designing and conducting workshops to advance the understanding of AI applications in SEO. My passion for continuous improvement and innovation makes me eager to share insights and strategies with peers at international SEO conferences, driving forward the conversation on how AI can transform our industry. Visit LinkedIn Profile Ozden Akyildiz  Paid Social Lead for EMEA AT Amazon UK Ozden has over a decade of experience in the digital marketing ecosystem, with a holistic point of view. She is passionate about creating and executing effective brand communication and strategy, leveraging digital marketing and social media platforms. Ozden holds a Facebook/Meta Media Buying & Planning Certification, demonstrating her proficiency and expertise in this domain. She possesses a deep knowledge and hands-on experience in analyzing data and drawing actionable insights for business objectives. Ozden has an advanced understanding of digital media KPIs and data-driven marketing, enabling her to optimize the performance and ROI of her campaigns. Additionally, she excels in team leadership, project planning and management, communication, and analytical skills, which she has developed through working with creative and media agencies. Ozden is always eager to learn new tools and techniques and to collaborate with diverse and talented professionals. Visit LinkedIn Profile SERBAY ARDA AYZIT Founder AT Insightus Marketing Consulting Visit LinkedIn Profile SERTAC SURMELI MANAGING PARTNER AT WOOHOO DIGITAL Sertac Surmeli is the co-founder and managing partner of Woohoo Digital, with over 15 years of experience in building product-focused digital solutions. With a strong background in UI/UX design and product management, he specializes in creating user-centered digital products that drive engagement and deliver real business value. His expertise includes strategic product development, innovation in design, and long-term digital planning. Sertac works closely with cross-functional teams to lead product-led growth and help organizations scale through smart, sustainable digital transformation. Visit LinkedIn Profile Ufuk Akcay COO-DatA and Analytics Consultant @ 3GEN Digital Growth Agency Ufuk Akçay is a data-driven analytics consultant with over 7 years of experience in the digital ecosystem. He currently serves as the COO at 3GEN Digital Growth Agency, where he leads projects in web and app analytics, CRO, SEO, and Performance Marketing. Throughout his career, he has worked with leading companies such as Migros, Boyner, and FLO and later co-founded several digital products, including GoCookie, GoComment, GoFeed, GoRedirect, and GoResize. His work focuses on driving sustainable growth through data-informed strategies and measurable results. Visit LinkedIn Profile Vanda Pokecz Head of SEO at Atolls Vanda is a Berlin based Head of SEO and SEO Product Lead at Atolls, a Certified Digital Product Manager and international speaker. She has spoken at prestigious events and podcasts like SMX Munich, BrightonSEO, International Search Summit Barcelona and WTSFest Berlin and has contributed to a number of publications. Starting her career within the product comparison sphere she quickly gained extensive experience in all areas of SEO and beyond. She ultimately found her true passion at the intersection of SEO and Product and advocates for the collaboration between disciplines for better user experience and improved business results. Currently Vanda as Head of SEO is part of an in-house SEO team helping millions of people around the World along their shopping and importantly savings journey. Visit LinkedIn Profile Yordan Dimitrov SEO Manager @ Reflect Digital Yordan Dimitrov is an SEO Manager at Reflect Digital, where he champions a human-first, data-driven approach to strategies that enhance visibility and ROI. With over six years of experience in SEO and Account Management, he has worked across various industries including leisure, e-commerce, and automotive, helping brands grow through a blend of technical expertise and creative insight. Born in neighbouring Bulgaria, Yordan moved to the UK at the age of 14 for his secondary education. He studied International Business Management at Coventry University, completing a placement year in the automotive sector that kickstarted his SEO journey. Since then, he has worked closely with clients across multiple sectors, developing commercially focused strategies that align with business goals. A keen advocate for SEO best practices, Yordan enjoys sharing insights on content strategy, technical SEO, and search trends. He has spoken at industry forums and conferences including SEO Office Hours, Majestic’s SEO in 2025: Additional Insights, and SERP Conf., contributing to discussions on the future of search and actionable SEO strategies. When he’s not analysing SERPs, he’s either capturing moments behind the camera or planning his next travel adventure. Visit LinkedIn Profile Stay tuned, more will be announced soon  ✨ Our Sponsors Platinum Sponsors BusinessUp! Majestic WhitePress Community Supporters 3GEN Digital Growth Agency Media Partners Dijital Ajanslar edvido GCS Network Global Fintech Market Search with Candour SEOFOMO SERP Conf. Take it Offline Workshop Sponsors inLinks Waikay Stay tuned, more will be announced soon  ✨  Want to become one of our sponsors? Click  here   to apply. Calling All Sponsors – Amplify Your Brand with Search ‘n Stuff! Your brand deserves to be in front of 200+ global decision-makers, industry leaders, and rising stars. As a sponsor, you’ll gain exposure through on-stage branding, social media, and premium event placements, reaching thousands through our newsletter campaigns and dedicated event promotions. Sponsorship Packages Include Benefits such as: Visibility on main stage branding, event materials & attendee handouts Targeted exposure to key decision-makers in SEO, PPC, eCommerce, and digital marketing Social media promotions & newsletter spotlights, delivering over 4M impressions CONTACT US FOR SPONSORSHIPS Why Should You Attend? Convince Your Manager or Colleagues! Need help making the case for your attendance? Here are  3 solid reasons  to join: Learn from the Best:  Gain exclusive insights from 24+ international speakers, including industry pioneers, technical experts, and growth strategists. Expect data-backed case studies, AI-driven strategies, and cutting-edge trends that will keep you ahead of the curve.   Unparalleled Networking Opportunities:  Meet 250+ digital marketers, founders, agency leaders, and in-house teams from around the world. Engage in structured speed networking, PowerPoint karaoke, live podcast sessions, and social events designed to help you build lasting connections and future collaborations. Career Growth & ROI:  Stay ahead of the competition by learning the latest in SEO, content, AI-driven marketing, eCommerce growth, CRO, paid search, and analytics. You’ll leave with actionable takeaways to improve workflows, enhance performance, and deliver measurable results to your company. Hands-On Learning in Workshops:  It’s not just about talks—our interactive workshops provide deep dives into technical SEO, analytics, conversion optimization, and more. Get hands-on experience with tools and techniques that can be applied immediately to your work. Enhance Your Brand Visibility:  Are you a freelancer, consultant, or looking to grow your personal brand? This is your chance to position yourself as an industry leader and get noticed by peers, potential clients, and hiring managers. A Unique 5-Star Experience:  Unlike traditional conferences, Search ‘n Stuff Antalya 2025 is set in a luxury, all-inclusive 5-star venue, Baia Lara Hotel. Enjoy networking by the beach, world-class amenities, and an unforgettable learning experience in a relaxe d yet professional environment. Need a template to convince your manager or team? We’ve got you covered! Download the free email template  here   to make your case. Frequently Asked Questions How can I make payment? You can purchase your ticket online via our secure payment system. If you need an invoice for company purchases, please email  yagmur@searchnstuff.co.uk. Do you offer a ny discounts for bulk purchases? Yes! We offer special discounts for group purchases.You can see the Group Tickets options for single and double room preferences. This means, if you are happy to share the room with your friend or colleague, you can prefer the Group Tickets with a double room. What is the refund policy if I am unable to attend? There will be no refunds for tickets that include accommodation, but we understand that plans can change. If you’re unable to attend, you can transfer your ticket to someone else until  September 30, 2025. Please note that name badges are pre-arranged and may not be adjustable after this date. For tickets without accommodation, refunds can be requested until July 1, 2025. After this date, all tickets will be non-refundable, but you will still have the option to transfer your ticket to another attendee. To request a refund (for non-accommodation tickets) or to transfer your ticket, please email us at yagmur@searchnstuff.co.uk as soon as possible with the necessary details. Are there recommended hotels nearby? Our 5-star conference venue, Baia Lara Hotel, includes luxury accommodation, meals, and drinks in select ticket types. If you’re looking for alternative hotels, we suggest booking nearby resorts early. Can I buy tickets in person on the day of the conference? No, tickets must be purchased in advance online. Due to venue capacity, we do not offer on-site ticket sales. Secure your spot early to avoid disappointment! What networking opportunities are available at the conference? This year, we’re elevating your networking experience with: ✔️ Speed Networking – Quick, structured meet-and-greet sessions. ✔️ PowerPoint Karaoke – Fun and unexpected presentation challenges. ✔️ Live Podcast Session – Be part of the conversation with industry experts. ✔️ Evening Socials & After-Parties – Unwind and connect in a relaxed setting. What’s the dress code? The conference dress code is smart casual. For evening networking events, we suggest business casual or chic casual—and don’t forget to bring something warm for Antalya’s cool October nights – just in case! Will Search ‘n Stuff Antalya 2025 offer a virtual / online experience? No, this is an in-person-only event. Our goal is to create an immersive experience that allows attendees to learn, connect, and engage in real life. However, select talks may be recorded for post-event access. Got more questions? We’re here to help! Reach out to us at info@searchnstuff.co.uk. About the Venue Baia Lara Hotel Nestled on the Mediterranean coast, this 5-star hotel offers luxury, comfort, and breathtaking views. Enjoy beach access, world-class dining, and exceptional amenities. DISCOVER THE VENUE Check out the highlights from our first-ever conference, organized in Antalya in October 2024 
 Subscribe to the newsletter to be the first to be informed! Stay in the loop with Search ‘n Stuff by signing up for our email list! Be the first to hear about early-bird discounts, exclusive updates, and all the exciting details for our upcoming events. Don’t miss your chance to connect, learn, and grow with us! 
 Share your love ❤️ 
 
 
 
 				 
 
 
 
 				 
 
 
 
 				 
Related Posts Search ‘n Stuff Antalya Conference 2024 December 4, 2024 
 	